# Preprocesamiento (Sleep-EDF-20)

Este notebook implementa el preprocesamiento sobre el subconjunto **Sleep-EDF-20**:
1. Localización y emparejamiento de archivos PSG / Hypnogram
2. Carga de señales con MNE
3. Preprocesamiento (filtrado, segmentación en épocas de 30s)
4. Codificación de etiquetas (vigilia vs. sueño)
5. División train/val/test por sujeto
6. Normalización

In [5]:
import os
import re
import glob
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

mne.set_log_level("WARNING")

DATA_DIR = os.path.expanduser("~/Desktop/TFM/datos")
print("Buscando datos en:", DATA_DIR)
print("Existe la ruta:", os.path.isdir(DATA_DIR))


Buscando datos en: C:\Users\alber/Desktop/TFM/datos
Existe la ruta: True


## 1. Localización y emparejamiento de archivos PSG / Hypnogram

Busca todos los `*-PSG.edf` y empareja cada uno con su `*-Hypnogram.edf` correspondiente (el sufijo de letra final varía: EC, EH, EJ, etc., así que emparejamos por el prefijo del sujeto/noche, ej. `SC4001`).

In [6]:
def find_psg_hypnogram_pairs(data_dir):
    psg_files = sorted(glob.glob(os.path.join(data_dir, "**", "*-PSG.edf"), recursive=True))
    pairs = []
    for psg in psg_files:
        base = os.path.basename(psg)
        # Prefijo tipo SC4001E -> subject+night, sin el último caracter (E0)
        prefix = base.split("-PSG.edf")[0][:-1]  # quita el '0' final de E0
        candidates = glob.glob(os.path.join(os.path.dirname(psg), prefix + "*-Hypnogram.edf"))
        if candidates:
            pairs.append((psg, candidates[0]))
        else:
            print(f"[AVISO] No se encontró Hypnogram para: {base}")
    return pairs

pairs = find_psg_hypnogram_pairs(DATA_DIR)
print(f"Pares PSG/Hypnogram encontrados: {len(pairs)}")
for psg, hyp in pairs[:5]:
    print(" -", os.path.basename(psg), "<->", os.path.basename(hyp))


Pares PSG/Hypnogram encontrados: 39
 - SC4001E0-PSG.edf <-> SC4001EC-Hypnogram.edf
 - SC4002E0-PSG.edf <-> SC4002EC-Hypnogram.edf
 - SC4011E0-PSG.edf <-> SC4011EH-Hypnogram.edf
 - SC4012E0-PSG.edf <-> SC4012EC-Hypnogram.edf
 - SC4021E0-PSG.edf <-> SC4021EH-Hypnogram.edf


## 2. Carga de señales con MNE

Mapeo de las anotaciones originales del hipnograma (W, 1, 2, 3, 4, R, ?, Movement) a las clases de interés. Por defecto se agrupa **vigilia (W) vs. sueño (todas las demás fases)** 

In [7]:
# Mapeo de anotaciones crudas del Sleep-EDF a clases binarias vigilia/sueño
ANNOTATION_TO_LABEL = {
    "Sleep stage W": 0,      # vigilia
    "Sleep stage 1": 1,      # sueño (N1)
    "Sleep stage 2": 1,      # sueño (N2)
    "Sleep stage 3": 1,      # sueño (N3)
    "Sleep stage 4": 1,      # sueño (N4, en R&K se fusiona con N3 en AASM)
    "Sleep stage R": 1,      # sueño REM
    # "Sleep stage ?": None,     # se descarta
    # "Movement time": None,     # se descarta
}

EPOCH_DURATION = 30.0  # segundos, según la resolución del hipnograma original

def load_subject(psg_path, hyp_path):
    raw = mne.io.read_raw_edf(psg_path, preload=True)
    annot = mne.read_annotations(hyp_path)
    raw.set_annotations(annot, emit_warning=False)

    sleep_onset = [a["onset"] for a in annot if a["description"] != "Sleep stage W"]
    if sleep_onset:
        start = max(raw.times[0], min(sleep_onset) - 30 * 60)
        end = min(raw.times[-1], max(sleep_onset) + 30 * 60)
        raw.crop(tmin=start, tmax=end)

    return raw

def raw_to_epochs_labels(raw, channel="EEG Fpz-Cz"):
    events, event_id = mne.events_from_annotations(
        raw, chunk_duration=EPOCH_DURATION
    )
    # Filtra solo los event_id que están en nuestro mapeo de interés
    valid_desc = {k: v for k, v in event_id.items() if k in ANNOTATION_TO_LABEL}
    if not valid_desc:
        return None, None

    epochs = mne.Epochs(
        raw, events, event_id=valid_desc,
        tmin=0, tmax=EPOCH_DURATION - 1.0 / raw.info["sfreq"],
        baseline=None, preload=True, picks=[channel]
    )
    X = epochs.get_data()  # (n_epochs, n_channels, n_samples)
    inv_id = {v: k for k, v in valid_desc.items()}
    y = np.array([ANNOTATION_TO_LABEL[inv_id[e]] for e in epochs.events[:, 2]])
    return X, y


## 3. Preprocesamiento: filtrado y construcción del dataset completo

Se aplica un filtro paso banda 0.5–30 Hz (rango habitual para actividad EEG relevante en sueño) y se recorre todos los sujetos, guardando también el ID de sujeto para poder hacer un split *subject-wise* más adelante.

In [8]:
ALL_X, ALL_y, ALL_subjects = [], [], []

for subj_idx, (psg_path, hyp_path) in enumerate(pairs):
    subject_name = os.path.basename(psg_path).split("-PSG.edf")[0]
    print(f"Procesando {subject_name} ({subj_idx + 1}/{len(pairs)})...")

    try:
        raw = load_subject(psg_path, hyp_path)
        raw.filter(l_freq=0.5, h_freq=30.0, picks=["EEG Fpz-Cz"])

        X, y = raw_to_epochs_labels(raw, channel="EEG Fpz-Cz")
        if X is None:
            print(f"  [AVISO] Sin épocas válidas para {subject_name}, se omite.")
            continue

        ALL_X.append(X)
        ALL_y.append(y)
        ALL_subjects.append(np.full(len(y), int(subject_name[3:5])))

    except Exception as e:
        print(f"  [ERROR] Fallo procesando {subject_name}: {e}")

X = np.concatenate(ALL_X, axis=0)          # (n_epochs_total, 1, n_samples)
y = np.concatenate(ALL_y, axis=0)
subjects = np.concatenate(ALL_subjects, axis=0)

print("\nForma final de X:", X.shape)
print("Forma final de y:", y.shape)
print("Distribución de clases -> vigilia (0):", (y == 0).sum(), "| sueño (1):", (y == 1).sum())


Procesando SC4001E0 (1/39)...


C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True)


Procesando SC4002E0 (2/39)...


C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True)


Procesando SC4011E0 (3/39)...


...

Procesando SC4191E0 (38/39)...


C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True)


Procesando SC4192E0 (39/39)...


C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True)
C:\Users\alber\AppData\Local\Temp\ipykernel_10760\85615995.py:16: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True)



Forma final de X: (73929, 1, 3000)
Forma final de y: (73929,)
Distribución de clases -> vigilia (0): 39906 | sueño (1): 34023


## 4. División train/val/test por sujeto

Crítico: las épocas de un mismo sujeto NO deben repartirse entre particiones distintas, o habría fuga de información (el modelo podría 'memorizar' patrones fisiológicos individuales en vez de aprender a distinguir vigilia/sueño en general).

In [9]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(X, y, groups=subjects))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(X[temp_idx], y[temp_idx], groups=subjects[temp_idx]))
val_idx, test_idx = temp_idx[val_idx], temp_idx[test_idx]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Sujetos en train:", len(set(subjects[train_idx])),
      "| val:", len(set(subjects[val_idx])),
      "| test:", len(set(subjects[test_idx])))


Train: (50005, 1, 3000) | Val: (11534, 1, 3000) | Test: (12390, 1, 3000)
Sujetos en train: 14 | val: 3 | test: 3


## 5. Normalización

Estandarización z-score, calculando media y desviación típica **solo con el conjunto de entrenamiento** para no filtrar información de val/test.

In [11]:
mean_train = X_train.mean()
std_train = X_train.std()

X_train_norm = (X_train - mean_train) / std_train
X_val_norm = (X_val - mean_train) / std_train
X_test_norm = (X_test - mean_train) / std_train

# Reordenar a (n_epochs, n_samples, n_channels) para Conv1D de Keras
X_train_norm = np.transpose(X_train_norm, (0, 2, 1))
X_val_norm = np.transpose(X_val_norm, (0, 2, 1))
X_test_norm = np.transpose(X_test_norm, (0, 2, 1))

print("Forma de entrada al modelo:", X_train_norm.shape)


Forma de entrada al modelo: (50005, 3000, 1)


In [21]:
import os

output_dir = os.path.expanduser("~/Desktop/TFM/datos_procesados")
os.makedirs(output_dir, exist_ok=True)  
np.savez(os.path.join(output_dir, "datos_procesados_1canal.npz"),
         X_train_norm=X_train_norm, y_train=y_train,
         X_val_norm=X_val_norm, y_val=y_val,
         X_test_norm=X_test_norm, y_test=y_test)

print("Guardado en:", os.path.join(output_dir, "datos_procesados_1canal.npz"))

Guardado en: C:\Users\alber/Desktop/TFM/datos_procesados\datos_procesados_1canal.npz


In [22]:
# ---- Comprobación del preprocesamiento y del reparto (1 canal) ----
sujetos_train = set(subjects[train_idx])
sujetos_val   = set(subjects[val_idx])
sujetos_test  = set(subjects[test_idx])

print("1. Forma de X (épocas, canales, muestras):", X.shape, "(esperado: (73929, 1, 3000))")
print("2. Sujetos distintos:", len(set(subjects)), "(esperado: 20)")
print("3. Sujetos en train / val / test:",
      len(sujetos_train), "/", len(sujetos_val), "/", len(sujetos_test), "(esperado: ~14 / 3 / 3)")
print("4. Sujetos repetidos entre conjuntos:",
      len(sujetos_train & sujetos_val) + len(sujetos_train & sujetos_test) + len(sujetos_val & sujetos_test),
      "(esperado: 0)")
print("5. Épocas en train / val / test:", len(train_idx), "/", len(val_idx), "/", len(test_idx),
      "| suma =", len(train_idx) + len(val_idx) + len(test_idx), "(esperado: 73929)")

for nombre, yy in [("train", y[train_idx]), ("val", y[val_idx]), ("test", y[test_idx])]:
    n0, n1 = (yy == 0).sum(), (yy == 1).sum()
    print(f"6. {nombre}: vigilia {n0} ({n0/len(yy):.1%}) | sueño {n1} ({n1/len(yy):.1%})")

print("7. NaN o Inf en X:", np.isnan(X).sum(), "/", np.isinf(X).sum(), "(esperado: 0 / 0)")
print("8. Media y desviación de train normalizado:",
      X_train_norm.mean().round(4), "/", X_train_norm.std().round(4), "(esperado: ~0 / ~1)")
print("   Forma X_train_norm:", X_train_norm.shape, "(esperado: (n, 3000, 1))")

1. Forma de X (épocas, canales, muestras): (73929, 1, 3000) (esperado: (73929, 1, 3000))
2. Sujetos distintos: 20 (esperado: 20)
3. Sujetos en train / val / test: 14 / 3 / 3 (esperado: ~14 / 3 / 3)
4. Sujetos repetidos entre conjuntos: 0 (esperado: 0)
5. Épocas en train / val / test: 50005 / 11534 / 12390 | suma = 73929 (esperado: 73929)
6. train: vigilia 26403 (52.8%) | sueño 23602 (47.2%)
6. val: vigilia 6590 (57.1%) | sueño 4944 (42.9%)
6. test: vigilia 6913 (55.8%) | sueño 5477 (44.2%)
7. NaN o Inf en X: 0 / 0 (esperado: 0 / 0)
8. Media y desviación de train normalizado: 0.0 / 1.0 (esperado: ~0 / ~1)
   Forma X_train_norm: (50005, 3000, 1) (esperado: (n, 3000, 1))
